# 第2回 宿題（発展）: 用語集を LoRA で学習し、学習前後と検索方式を比べる

演習のノートブック（`notebooks/lora_format_tuning.ipynb`）をもとに、題材を架空 FAQ から **講座の用語集**（`data/glossary.json`・58 語）に替えたものです。

**演習との違い**

| 項目 | 演習（FAQ） | このノートブック（用語集） |
| --- | --- | --- |
| 入力 | FAQ の全文 ＋ 質問 | 質問だけ（用語集は渡さない＝定義をモデルに覚えさせる） |
| 学習データ | 14 件 | 132 件（用語集の 58 語 × 言い回し 2 種 ＋ 用語集に無い語 8 語 × 2 種） |
| 評価 | 21 問 | 26 問（学習と同じ言い回し 8・言い換え 10・用語集に無い語 8） |
| 出力 | `{"answer", "needs_review"}` の JSON 1 行 | 同じ（用語集に無い語は `needs_review: true`） |

学習データと評価セットは `scripts/make_sft_dataset.py` で作り、セクション 2 でアップロードします。

```bash
uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl
uv run python scripts/make_sft_dataset.py --eval > glossary_eval.json
```

**所要時間の目安（T4）**: 0〜3 が約 8 分、学習が 15〜20 分、学習後評価と提出が約 5 分。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

演習の `CONFIG` から変えたのは次の 5 つです（他は同じ）。

- `num_train_epochs: 10`: 学習データ 132 件を 10 周。1 周が 17 ステップなので合計 170 ステップです。
- `warmup_steps: 10`: ステップ数が増えたのに合わせて少し伸ばしました。
- `max_new_tokens: 320`: 用語の定義は FAQ の回答より長い（最長 212 字）ので、途中で切れて JSON が壊れないように増やしました。
- `max_seq_len: 768`: system プロンプトと長い定義を合わせると 512 トークンに近づくので、学習時に末尾（回答の終わり）が切り捨てられないように増やしました。
- `work_dir`: 演習の結果と混ざらないよう別のフォルダにしました。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 10,  # 学習データ（132 件）を何周学習するか。1 周 17 ステップなので合計 170 ステップ
    "warmup_steps": 10,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 768,     # 長い定義でも回答の終わりまで学習させる
    "max_new_tokens": 320,   # 定義は FAQ の回答より長いので増やす
    "gen": {"do_sample": False},
    "work_dir": "/content/glossary_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 学習データと評価セットの読み込み

**やること**: セルを実行すると「ファイル選択」ボタンが出るので、`glossary_sft.jsonl` と `glossary_eval.json` の 2 つを選んでアップロードします（すでに `/content/glossary_lab/` にあれば聞かれません）。

- 学習データの各行は `messages`（system / user / assistant）形式なので、そのまま `Dataset.from_list` に渡せます。
- 評価に使う system プロンプトは評価セットのファイルから読み、学習データの system と同じであることを確かめます。
- 評価の 26 問は 3 種類です。`same` は学習と同じ言い回し（覚えたかの確認）、`reworded` は学習に無い言い回しや別名（言い換えへの汎化）、`ungrounded` は用語集に無い語（`needs_review: true` が正解）です。

**見るところ**: `train=132 ... test=26 gold=26` と、種類ごとの件数が表示されます。

In [ ]:
from collections import Counter

DATA_FILES = ["glossary_sft.jsonl", "glossary_eval.json"]
if any(not (WORK / n).exists() for n in DATA_FILES):
    from google.colab import files
    for name, data in files.upload().items():
        # 同名のファイルが既にあると Colab は「名前 (1).拡張子」で保存するので、元の名前に戻して置く
        base = next((n for n in DATA_FILES if name.startswith(Path(n).stem) and name.endswith(Path(n).suffix)), name)
        (WORK / base).write_bytes(data)
missing = [n for n in DATA_FILES if not (WORK / n).exists()]
assert not missing, f"アップロードされていないファイルがあります: {missing}"

train_records = [json.loads(line) for line in (WORK / "glossary_sft.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
EVAL = json.loads((WORK / "glossary_eval.json").read_text(encoding="utf-8"))
SYSTEM = EVAL["system"]
test, gold = EVAL["test"], EVAL["gold"]
assert all(r["messages"][0]["content"] == SYSTEM for r in train_records), "学習データと評価セットで system プロンプトが違います"

n_unknown = sum(json.loads(r["messages"][2]["content"])["needs_review"] for r in train_records)
print(f"train={len(train_records)}（用語集にある語 {len(train_records) - n_unknown}・無い語 {n_unknown}） test={len(test)} gold={len(gold)}")
print("評価の種類:", dict(Counter(r["category"] for r in test)))
print("学習例の 1 件 | 質問:", train_records[0]["messages"][1]["content"], "| 望ましい出力:", train_records[0]["messages"][2]["content"][:80], "...")

## 3. モデルの読み込みと学習前評価

演習と同じモデルを量子化なしで読み込み、26 問に答えさせて採点します。入力は system プロンプトと質問だけで、用語集は渡しません。

**採点のしかた**（演習と同じ考え方）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、正解の語句（定義の中の要の語。例: LoRA なら「低ランク」）が `answer` に含まれること。
- **定義との一致度**（参考）: 用語集にある語について、`answer` と用語集の定義の文字列の近さ（0〜1、`difflib`）。覚えた定義をどこまでそのまま再現したかの目安です。
- **失敗の型**: 形式の崩れ／用語集に無い語に定義を作って断定した／用語集にある語に「確認が必要」と逃げた／`needs_review` は合っているが要の語が無い（定義の誤り）。

**見るところ**: 種類ごとの内容正解率（`content_same`・`content_reworded`・`content_ungrounded`）。学習前は、一般的な語（トークン・LoRA など）はモデルの元の知識で答えられることがある一方、講座に固有の語（宿題の3レベル・lecNN-start タグ・既定モデルなど）は答えられないはずです。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# SYSTEM はセクション 2 で評価セットから読んだもの（学習データと同じ）。用語集は入力に渡さない
def build_messages(example):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": example["question"]}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re, difflib
import pandas as pd
pd.set_option("display.max_colwidth", None)

CATEGORIES = ["same", "reworded", "ungrounded"]

def check_format(text: str):
    """JSON として読め、必須キーと型が合うか。失敗理由も返す"""
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    """内容の合否: needs_review が正解と一致し、必須語句を answer に含む"""
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in str(obj.get("answer", "")) for w in gold_entry["must_include"])

def failure_type(ok_fmt, obj, gold_entry, ok_content):
    if ok_content: return "" if ok_fmt else "形式の崩れ"
    if obj is None: return "形式の崩れ"
    if gold_entry["needs_review"] and obj.get("needs_review") is False: return "断定（用語集に無い語）"
    if not gold_entry["needs_review"] and obj.get("needs_review") is True: return "確認要求で逃げた"
    return "定義の誤り（要の語が無い）"

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    """26 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する"""
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        loose = loose if isinstance(loose, dict) else None
        g = gold[ex["question"]]
        ok_content = check_content(loose, g)
        sim = None
        if ex["definition"] and loose is not None:
            sim = round(difflib.SequenceMatcher(None, str(loose.get("answer", "")), ex["definition"]).ratio(), 2)
        rows.append({"question": ex["question"], "category": ex["category"], "raw": raw,
                     "format_ok": ok_fmt, "format_reason": reason, "content_ok": ok_content,
                     "both_ok": ok_fmt and ok_content, "failure": failure_type(ok_fmt, loose, g, ok_content),
                     "def_similarity": sim, "gold_needs_review": g["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": round(sum(r["format_ok"] for r in rows)/n, 3),
               "content_rate": round(sum(r["content_ok"] for r in rows)/n, 3),
               "both_rate": round(sum(r["both_ok"] for r in rows)/n, 3)}
    for c in CATEGORIES:
        sub = [r for r in rows if r["category"] == c]
        summary[f"content_{c}"] = round(sum(r["content_ok"] for r in sub)/len(sub), 3)
    sims = [r["def_similarity"] for r in rows if r["def_similarity"] is not None]
    summary["def_similarity_mean"] = round(sum(sims)/len(sims), 2) if sims else None
    summary["failures"] = dict(Counter(r["failure"] for r in rows if r["failure"]))
    summary["seconds"] = round(time.time()-t0, 1)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"種類": r["category"], "質問": r["question"], "学習前の出力": r["raw"], "形式": mark(r["format_ok"]),
                       "内容": mark(r["content_ok"]), "失敗の型": r["failure"], "定義との一致度": r["def_similarity"]} for r in before_rows]))

## 4. LoRA 学習（T4 で 15〜20 分）

演習と同じ LoRA の設定（`target_modules`・バッチ 4 × 勾配累積 2）で、`glossary_sft.jsonl` の 132 件を学習します。検証データは使いません（評価はセクション 5 の 26 問で行います）。

**見るところ**

- `Training Loss`（10 ステップごと）が下がっていくこと。定義の文章を丸ごと覚える必要があるので、演習（0.01〜0.02 まで下がる）より下がり方が遅い可能性があります。最後の値を記録してください。
- `学習時間: ... 秒 | ステップ数: 170 | エポック: 10.0`。

**途中で止まったとき**: チェックポイントは 50 ステップごとに `/content/glossary_lab/ckpt/` に保存されます。演習の 4b と同じく、0〜3 を実行し直してから `trainer.train(resume_from_checkpoint=True)` で続きを学習できます。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)   # 各行が messages 形式なので、そのまま渡せる

model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=50, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time()-t0)
FINAL_LOSS = next((h["loss"] for h in reversed(trainer.state.log_history) if "loss" in h), None)
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | 最終 loss: {FINAL_LOSS}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後評価（学習前と同じ 26 問・同じ生成設定）

**見るところ**

- 種類ごとの内容正解率: `same`（覚えたか）・`reworded`（言い換えに効くか）・`ungrounded`（知らない語に「確認が必要」と言えるか）がそれぞれどう変わったか。
- 定義との一致度: 学習後に定義をどこまでそのまま返すようになったか。
- 失敗の型: 学習前後でどの型が増え、どの型が減ったか。特に、用語集に無い語に、それらしい定義を作って断定していないか。
- `退行`: 学習前は通っていたのに、学習後に落ちた質問。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
cols = ["n", "format_rate", "content_rate", "content_same", "content_reworded", "content_ungrounded", "def_similarity_mean", "seconds"]
display(pd.DataFrame([before_summary, after_summary]).set_index("label")[cols])
print("失敗の型 | 学習前:", before_summary["failures"], "| 学習後:", after_summary["failures"])
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

detail = pd.DataFrame([{"種類": b["category"], "質問": b["question"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}", "学習前 失敗の型": b["failure"],
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}", "学習後 失敗の型": a["failure"],
                        "一致度 前→後": f"{b['def_similarity']}→{a['def_similarity']}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 比較表と提出

**やること**: 設定、ライブラリの版、学習の記録、学習前後の要約、退行した質問を `result.json` にまとめます。

**ダウンロードするもの**（左のファイル一覧の `/content/glossary_lab/` で右クリック → ダウンロード）: `result.json`、`eval_before.json`、`eval_after.json`

終わったら **ランタイム → ランタイムを接続解除して削除** を実行します。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）・用語集 SFT",
    "data": {"train": len(train_records), "train_unknown_terms": n_unknown, "test": dict(Counter(r["category"] for r in test))},
    "training": {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "final_loss": FINAL_LOSS},
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])